# Multiple-Break Detection in the FRED-MD Dataset

We apply the proposed method to the FRED-MD (Federal Reserve Economic Data – Monthly Data) dataset to illustrate its empirical performance in detecting structural breakpoints in macroeconomic time series. The FRED database is maintained by the Research Division of the Federal Reserve Bank of St. Louis, and the data are publicly available and continuously updated. The sample used in this study consists of 126 unbalanced monthly macroeconomic time series spanning from January 1959 to November 2025. According to the standard FRED-MD classification, these series are grouped into eight economic categories: output and income, labor market, housing, consumption (including orders and inventories), money and credit, interest and exchange rates, prices, and the stock market.


In [1]:
import numpy as np
import pandas as pd

from fred.remove_outliers import remove_outliers
from fred.prepare_missing import prepare_missing
from fred.factors_em import factors_em
from fmbqml import MultiBreakQML


Using the FRED-MD macroeconomic database, this study first preprocesses the raw time-series data. Missing values in the original dataset are imputed using the expectation-maximization (**EM**) algorithm. Subsequently, variable-specific transformation rules, such as differencing or log-differencing, are applied to obtain approximately stationary time series, and outlying observations are adjusted accordingly. After these preprocessing steps, each macroeconomic variable contains a balanced monthly sample of length (T=801). Next, the information criteria proposed by Bai and Ng (2002) are employed to determine the optimal number of common factors within a prespecified maximum-factor range; in this study, the **IC2** criterion is used. Once the number of factors is selected, the latent common factors are estimated using a principal components approach combined with the **EM** algorithm, which yields stable factor series even in the presence of missing observations. The resulting factor matrix is then organized with time as the index and used for subsequent econometric analysis. For detailed descriptions of the construction of the FRED-MD dataset and the variable transformation procedures, see McCracken and Ng (2016). The first five observations of the processed data are also reported in the paper for illustration.


In [2]:
input_csv = "data/FRED_MD_2025_11.csv"

demean_method = 2
factor_ic_method = 2
max_factors = 8
min_segment_length = 20
max_breaks = 7

# ========================================================================
# PART 1: LOAD AND LABEL DATA
# ========================================================================

fred_df = pd.read_csv(input_csv)

transform_codes = fred_df.iloc[0, 1:].astype(float).astype(int).to_numpy()
raw_data = fred_df.iloc[1:, 1:].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)

date_column = fred_df.columns[0]
date_strings = fred_df[date_column].astype(str)

final_date = pd.to_datetime(date_strings.iloc[-1], errors="coerce")
if pd.isna(final_date):
    raise ValueError("Cannot parse final date from CSV first column.")

final_month = final_date.month
final_year = final_date.year

monthly_dates = pd.date_range(
    start="1959-01-01",
    end=f"{final_year:04d}-{final_month:02d}-01",
    freq="MS"
)

num_raw_dates = len(monthly_dates)
raw_data = raw_data[:num_raw_dates, :]

# ========================================================================
# PART 2: PROCESS DATA
# ========================================================================

transformed_data = prepare_missing(raw_data, transform_codes)

transformed_data = transformed_data[2:num_raw_dates, :]

clean_data, outlier_counts = remove_outliers(transformed_data)

# ========================================================================
# PART 3: EM IMPUTATION
# ========================================================================

residuals, em_factors, loadings, eigenvalues, filled_data = factors_em(
    clean_data,
    max_factors,
    factor_ic_method,
    demean_method
)


# ========================================================================
# PART 4: FULL-SAMPLE STANDARDIZATION
# ========================================================================

em_filled_data = filled_data
sample_size, num_series = em_filled_data.shape

column_means = np.mean(em_filled_data, axis=0)
column_stds = np.std(em_filled_data, axis=0, ddof=1)
column_stds[column_stds == 0] = 1.0

standardized_data = (em_filled_data - column_means) / column_stds

print("\n============================================================")
print("FRED-MD empirical application with MultiBreakQML")
print(f"Vintage file: {input_csv}")
print(f"After EM + outlier replacement: T = {sample_size}, N = {num_series}")
print(
    f"Minimum segment length h = {min_segment_length} months, "
    f"trim_ratio = {min_segment_length / sample_size:.4f}"
)
print(f"Candidate #breaks: m = 0,1,...,{max_breaks}")
print("============================================================\n")


# ========================================================================
# PART 5: CREATE CALENDAR INDEX
# ========================================================================

start_year = 1959
start_month = 3

calendar_labels = []
for time_index in range(1, sample_size + 1):
    month_offset = (start_month - 1) + (time_index - 1)
    year = start_year + month_offset // 12
    month = month_offset % 12 + 1
    calendar_labels.append(f"{year:04d}-{month:02d}")



Iteration 1: obj 999.000000 IC 7
Iteration 2: obj   0.005356 IC 7
Iteration 3: obj   0.001614 IC 7
Iteration 4: obj   0.000944 IC 7
Iteration 5: obj   0.000676 IC 7


Iteration 6: obj   0.000514 IC 7
Iteration 7: obj   0.000400 IC 7
Iteration 8: obj   0.000317 IC 7
Iteration 9: obj   0.000254 IC 7
Iteration 10: obj   0.000207 IC 7


Iteration 11: obj   0.000170 IC 7
Iteration 12: obj   0.000141 IC 7
Iteration 13: obj   0.000117 IC 7
Iteration 14: obj   0.000098 IC 7
Iteration 15: obj   0.000082 IC 7


Iteration 16: obj   0.000069 IC 7
Iteration 17: obj   0.000058 IC 7
Iteration 18: obj   0.000049 IC 7
Iteration 19: obj   0.000042 IC 7
Iteration 20: obj   0.000036 IC 7


Iteration 21: obj   0.000030 IC 7
Iteration 22: obj   0.000026 IC 7
Iteration 23: obj   0.000022 IC 7
Iteration 24: obj   0.000019 IC 7
Iteration 25: obj   0.000017 IC 7


Iteration 26: obj   0.000014 IC 7
Iteration 27: obj   0.000012 IC 7
Iteration 28: obj   0.000011 IC 7
Iteration 29: obj   0.000009 IC 7
Iteration 30: obj   0.000008 IC 7


Iteration 31: obj   0.000007 IC 7
Iteration 32: obj   0.000006 IC 7
Iteration 33: obj   0.000006 IC 7
Iteration 34: obj   0.000005 IC 7
Iteration 35: obj   0.000004 IC 7


Iteration 36: obj   0.000004 IC 7
Iteration 37: obj   0.000003 IC 7
Iteration 38: obj   0.000003 IC 7
Iteration 39: obj   0.000003 IC 7
Iteration 40: obj   0.000002 IC 7


Iteration 41: obj   0.000002 IC 7
Iteration 42: obj   0.000002 IC 7
Iteration 43: obj   0.000002 IC 7
Iteration 44: obj   0.000001 IC 7
Iteration 45: obj   0.000001 IC 7


Iteration 46: obj   0.000001 IC 7
Iteration 47: obj   0.000001 IC 7

FRED-MD empirical application with MultiBreakQML
Vintage file: data/FRED_MD_2025_11.csv
After EM + outlier replacement: T = 801, N = 126
Minimum segment length h = 20 months, trim_ratio = 0.0250
Candidate #breaks: m = 0,1,...,7



After obtaining the EM-imputed and standardized panel, the `MultiBreakQML` procedure is employed to detect structural changes in its latent factor structure. Specifically, the standardized panel is used to construct a `MultiBreakQML` object, with the maximum number of allowable breakpoints set to `max_break=7`, the upper bound on the number of candidate factors set to `max_factors=8`, `factor_criterion=1` (i.e., **IC2**) adopted for factor-dimension selection, and `trim_ratio=20/T` corresponding to a minimum segment length of approximately 20 observations. Subsequently, the method `estimate_breaks_jointly(min_break=0, classify=True, verbose=True)` is called to jointly estimate the breakpoint locations and model dimension. Here, setting `min_break=0` allows for the possibility of no structural break, so the procedure can simultaneously determine whether structural breaks are present in the data. The object-level `trim_ratio` parameter controls the minimum admissible segment length used in model selection and breakpoint determination. Since `classify=True`, the method also classifies the type of each estimated breakpoint. Ultimately, the procedure returns the estimated set of breakpoints, the corresponding number of breakpoints, and the type of each breakpoint, thereby providing a basis for subsequent segmented analysis and the study of structural changes.


In [3]:
qml_panel = pd.DataFrame(standardized_data, index=calendar_labels)

model = MultiBreakQML(
    qml_panel,
    max_break=max_breaks,
    max_factors=max_factors,
    factor_criterion="IC2",
    trim_ratio=min_segment_length / len(qml_panel)
)

result = model.estimate_breaks_jointly(
    min_break=0,
    classify=True,
    verbose=True
)

MultiBreakQML JOINT ESTIMATION SUMMARY
Test type: mqml_joint
Sample size: T = 801, N = 126
Allowed breaks: max=7, min=0
Max factors: 8, trim_ratio=0.024968789013732832 (min_segment_length=20)
Estimated number of factors: 7
Detected breakpoints: [119, 287, 592, 613, 732]
Break types: ['Type 1 (Singular)', 'Type 1 (Singular)', 'Type 1 (Singular)', 'Type 1 (Singular)', 'Type 1 (Singular)']
Breakpoint labels: ['1969-01', '1983-01', '2008-06', '2010-03', '2020-02']
Number of breaks: 5
Factor criterion: IC2

Information criterion path:
m=0: U_NT=0.0000, IC=0.0000, breakpoints=[]
m=1: U_NT=-1018.0253, IC=-554.3334, breakpoints=[556]
m=2: U_NT=-1459.3716, IC=-531.9878, breakpoints=[554, 732]
m=3: U_NT=-1982.1257, IC=-591.0501, breakpoints=[590, 613, 732]
m=4: U_NT=-2418.0025, IC=-563.2351, breakpoints=[300, 590, 613, 732]
m=5: U_NT=-2923.7754, IC=-605.3161, breakpoints=[119, 287, 592, 613, 732]
m=6: U_NT=-3210.9225, IC=-428.7713, breakpoints=[117, 336, 493, 589, 613, 732]
m=7: U_NT=-3494.0214,

In [4]:
print("\nFinal result:")
print("Number of breaks:", result["n_breaks"])
print("Breakpoints:", result["breakpoints"])
print("Break dates:", result["break_labels"])
print("Break types:", result.get("break_types"))
print("Regime factors:", result.get("regime_factors"))
print("Combined factors:", result.get("combined_factors"))




Final result:
Number of breaks: 5
Breakpoints: [119, 287, 592, 613, 732]
Break dates: ['1969-01', '1983-01', '2008-06', '2010-03', '2020-02']
Break types: ['Type 1 (Singular)', 'Type 1 (Singular)', 'Type 1 (Singular)', 'Type 1 (Singular)', 'Type 1 (Singular)']
Regime factors: [2, 5, 7, 3, 4, 7]
Combined factors: [6, 7, 7, 5, 7]


## Sensitivity to the maximum number of breaks

To assess whether the empirical break estimates depend on the imposed upper bound, the joint estimation is repeated for `max_break` equal to 4, 5, 6, and 7. The factor-number bound, IC2 criterion, minimum segment length, data vintage, and preprocessing steps are held fixed. The resulting table reports the selected number of breaks and their calendar dates.


In [5]:
from pathlib import Path

max_break_grid = (4, 5, 6, 7)
max_break_sensitivity_results = {}
max_break_sensitivity_rows = []

for candidate_max_break in max_break_grid:
    if candidate_max_break == max_breaks:
        sensitivity_result = result
    else:
        sensitivity_model = MultiBreakQML(
            qml_panel,
            max_break=candidate_max_break,
            max_factors=max_factors,
            factor_criterion="IC2",
            trim_ratio=min_segment_length / len(qml_panel),
        )
        sensitivity_result = sensitivity_model.estimate_breaks_jointly(
            min_break=0,
            classify=False,
            verbose=False,
        )

    max_break_sensitivity_results[candidate_max_break] = sensitivity_result
    estimated_dates = sensitivity_result.get("break_labels", [])
    max_break_sensitivity_rows.append({
        "max_break": candidate_max_break,
        "Selected number of breaks": sensitivity_result.get("n_breaks"),
        "Estimated break dates": ", ".join(map(str, estimated_dates)),
    })

max_break_sensitivity_table = pd.DataFrame(max_break_sensitivity_rows)
display(max_break_sensitivity_table)

table_output_dir = Path("output") / "tables"
table_output_dir.mkdir(parents=True, exist_ok=True)
csv_path = table_output_dir / "fred_md_max_break_sensitivity.csv"
latex_path = table_output_dir / "fred_md_max_break_sensitivity.tex"

max_break_sensitivity_table.to_csv(csv_path, index=False)
max_break_sensitivity_table.to_latex(
    latex_path,
    index=False,
    caption="Sensitivity of FRED-MD break estimates to the maximum number of breaks.",
    label="tab:fred_md_max_break_sensitivity",
)

print(f"CSV table saved to: {csv_path}")
print(f"LaTeX table saved to: {latex_path}")

,max_break,Selected number of breaks,Estimated break dates
0,4,3,"2008-04, 2010-03, 2020-02"
1,5,5,"1969-01, 1983-01, 2008-06, 2010-03, 2020-02"
2,6,5,"1969-01, 1983-01, 2008-06, 2010-03, 2020-02"
3,7,5,"1969-01, 1983-01, 2008-06, 2010-03, 2020-02"


CSV table saved to: output\tables\fred_md_max_break_sensitivity.csv
LaTeX table saved to: output\tables\fred_md_max_break_sensitivity.tex
